# Experiment 7: Semi-Supervised Learning (Self-Training Classifier)

**Objective:**
1. Simulate a realistic semi-supervised scenario where only 20% of data is labeled and 80% is unlabeled.
2. Train a baseline supervised `LogisticRegression` on only the labeled 20%.
3. Train a semi-supervised `SelfTrainingClassifier` that iteratively assigns pseudo-labels to confident unlabeled points.
4. Compare accuracy and count pseudo-labeled samples.

### Step 1: Import Libraries

In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.semi_supervised import SelfTrainingClassifier
from sklearn.metrics import accuracy_score

### Step 2: Generate Classification Dataset

In [2]:
# Generate 500 samples with 2 informative features
X, y = make_classification(
    n_samples=500, n_features=2, n_informative=2,
    n_redundant=0, n_clusters_per_class=1,
    random_state=42
)

# 80% Train, 20% Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Total training samples: {len(X_train)}")
print(f"Total testing samples : {len(X_test)}")

Total training samples: 400
Total testing samples : 100


### Step 3: Simulate Partially Labeled Data (Mask 80% as -1)

In [3]:
rng = np.random.RandomState(42)
y_semi = y_train.copy()

# Randomly mark 80% of training data as unlabeled (-1)
unlabeled_indices = rng.choice(
    len(y_train), size=int(0.8 * len(y_train)), replace=False
)
y_semi[unlabeled_indices] = -1

labeled_mask = (y_semi != -1)
print(f"Labeled training samples  : {np.sum(labeled_mask)} (20%)")
print(f"Unlabeled training samples: {np.sum(~labeled_mask)} (80%)")

Labeled training samples  : 80 (20%)
Unlabeled training samples: 320 (80%)


### Step 4: Baseline Supervised Model (Only 20% Labeled Data)

In [4]:
# Train Logistic Regression on only labeled points
baseline_model = LogisticRegression()
baseline_model.fit(X_train[labeled_mask], y_semi[labeled_mask])

# Evaluate on test set
baseline_pred = baseline_model.predict(X_test)
baseline_acc = accuracy_score(y_test, baseline_pred)

print(f"Supervised Baseline Accuracy (20% data): {baseline_acc * 100:.2f}%")

Supervised Baseline Accuracy (20% data): 88.00%


### Step 5: Semi-Supervised Model (Self-Training Classifier)

In [5]:
# SelfTrainingClassifier pseudo-labels points with probability >= threshold (0.8)
self_training_model = SelfTrainingClassifier(
    estimator=LogisticRegression(),
    threshold=0.8,
    max_iter=10
)

# Fit on all training data (including unlabeled points marked -1)
self_training_model.fit(X_train, y_semi)

# Evaluate on test set
self_pred = self_training_model.predict(X_test)
self_acc = accuracy_score(y_test, self_pred)

# Count how many samples were pseudo-labeled
pseudo_labeled_count = np.sum(self_training_model.transduction_ != -1) - np.sum(labeled_mask)

print(f"Self-Training Model Accuracy           : {self_acc * 100:.2f}%")
print(f"Pseudo-labeled samples added           : {pseudo_labeled_count} / {np.sum(~labeled_mask)}")

Self-Training Model Accuracy           : 89.00%
Pseudo-labeled samples added           : 281 / 320


### Step 6: Summary Comparison

In [6]:
print("=== Model Comparison ===")
print(f"1. Supervised Baseline (20% Labeled)   : {baseline_acc * 100:.2f}%")
print(f"2. Self-Training Model (Semi-Supervised): {self_acc * 100:.2f}%")
print(f"Accuracy Gain: +{(self_acc - baseline_acc) * 100:.2f}%")

=== Model Comparison ===
1. Supervised Baseline (20% Labeled)   : 88.00%
2. Self-Training Model (Semi-Supervised): 89.00%
Accuracy Gain: +1.00%
